<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/llm-agents/02_langchain_arac_entegrasyonu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain ve Araç (Tool) Entegrasyonu

**LangChain**, LLM tabanlı uygulamaları parça parça (model, istem şablonu, çıktı ayrıştırıcı, retriever, araç, ajan) birleştirerek kurmayı sağlayan bir çatıdır. En büyük avantajı **sağlayıcıdan bağımsız** olmasıdır: aynı kod Ollama ile yerel bir modelde de, herhangi bir bulut API'sinde de çalışır; sadece model nesnesi değişir.

| # | Bölüm | Ne öğreneceğiz? |
|---|---|---|
| 1 | Kurulum | Colab'da **ücretsiz, yerel** LLM: Ollama + Qwen2.5 |
| 2 | Mesajlar ve model çağrısı | `invoke`, `stream`, `batch` |
| 3 | İstem şablonları | `ChatPromptTemplate` |
| 4 | LCEL zincirleri | `prompt \| llm \| parser`, paralel ve özel adımlar |
| 5 | Yapılandırılmış çıktı | Pydantic şeması ile güvenilir JSON |
| 6 | Araç tanımlama | `@tool` dekoratörü, şema, doğrudan çağırma |
| 7 | Tool calling döngüsü | `bind_tools` ile modelin araç seçmesi, döngüyü elle yazmak |
| 8 | Ajan | `create_agent` ile hazır ajan döngüsü |
| 9 | Hafıza | Checkpointer ile çok turlu sohbet |
| 10 | RAG zinciri ve retriever aracı | Vektör deposu + zincir, ajanın aracı olarak arama |

> **Tamamen ücretsiz, API anahtarı yok.** Model Colab'ın GPU'sunda Ollama ile çalışır. `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçin.

---
## 1) Kurulum: Ollama ile yerel LLM

[Ollama](https://ollama.com), açık kaynak modelleri tek komutla indirip yerel bir sunucu olarak çalıştırır. **Qwen2.5-7B-Instruct** Türkçeyi iyi anlar, **tool calling** destekler ve 4-bit kuantize hali (~4.7 GB) T4 GPU'ya rahatça sığar.

In [ ]:
!sudo apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U langchain langchain-ollama langchain-community langchain-huggingface langchain-text-splitters langgraph sentence-transformers

In [ ]:
import subprocess, time, requests

ollama_sureci = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):                                 # sunucu ayağa kalkana kadar bekle
    try:
        requests.get("http://localhost:11434", timeout=1)
        print("Ollama sunucusu çalışıyor ✅")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

In [ ]:
MODEL = "qwen2.5:7b"          # daha hızlı ama daha zayıf alternatif: "qwen2.5:3b"
!ollama pull {MODEL}

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL, temperature=0)
print(llm.invoke("Merhaba! Kendini bir cümleyle tanıt.").content)

> **Başka bir modele geçmek:** LangChain'in sağlayıcıdan bağımsız arayüzü sayesinde defterin geri kalanı değişmeden çalışır. Ör. `ChatOllama(model="llama3.1:8b")`, `ChatOllama(model="mistral-nemo")` veya ücretsiz katmanı olan bir bulut sağlayıcısının LangChain entegrasyonu.

---
## 2) Mesajlar ve model çağrısı

Sohbet modelleri **mesaj listesi** alır:

| Mesaj tipi | Rol | Ne için? |
|---|---|---|
| `SystemMessage` | system | Modelin rolü, kuralları, üslubu |
| `HumanMessage` | user | Kullanıcının mesajı |
| `AIMessage` | assistant | Modelin cevabı (ve araç çağrı istekleri) |
| `ToolMessage` | tool | Bir aracın çalıştırılma sonucu |

Her LangChain bileşeni (model, zincir, ajan) aynı **Runnable** arayüzünü paylaşır: `invoke` (tek), `stream` (parça parça), `batch` (toplu).

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

mesajlar = [
    SystemMessage("Sen sabırlı bir yapay zeka öğretmenisin. Lise öğrencisine anlatır gibi kısa ve net anlat."),
    HumanMessage("Overfitting nedir?"),
]
cevap = llm.invoke(mesajlar)
print(cevap.content)
print("\nToken kullanımı:", cevap.usage_metadata)

In [ ]:
# stream: cevabı üretildikçe parça parça al (sohbet arayüzleri böyle çalışır)
for parca in llm.stream("Gradient descent'i 3 maddede açıkla."):
    print(parca.content, end="", flush=True)

In [ ]:
# batch: birden fazla isteği tek seferde gönder
sorular = ["Bagging nedir? Tek cümle.", "Boosting nedir? Tek cümle.", "Dropout nedir? Tek cümle."]
for soru, c in zip(sorular, llm.batch(sorular)):
    print(f"• {soru}\n  {c.content}\n")

---
## 3) İstem şablonları (Prompt Templates)

Değişkenli, tekrar kullanılabilir istemler. Değişkenler `{süslü_parantez}` ile belirtilir.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ceviri_sablonu = ChatPromptTemplate.from_messages([
    ("system", "Sen profesyonel bir çevirmensin. Metni {hedef_dil} diline çevir. Sadece çeviriyi yaz."),
    ("human", "{metin}"),
])

hazir_istem = ceviri_sablonu.invoke({"hedef_dil": "İngilizce", "metin": "Yapay zeka hayatımızı değiştiriyor."})
print("Oluşan mesajlar:")
for m in hazir_istem.to_messages():
    print(f"  [{m.type}] {m.content}")

---
## 4) LCEL: LangChain Expression Language

Bileşenler `|` (pipe) operatörüyle zincirlenir; bir adımın çıktısı bir sonrakinin girdisi olur. Ortaya çıkan zincir de bir Runnable'dır (`invoke`, `stream`, `batch` hepsi çalışır).

```
{"hedef_dil": ..., "metin": ...} ──► ChatPromptTemplate ──► ChatOllama ──► StrOutputParser ──► "çeviri metni"
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser

ceviri_zinciri = ceviri_sablonu | llm | StrOutputParser()

print(ceviri_zinciri.invoke({"hedef_dil": "İngilizce", "metin": "Bugün hava çok güzel, parka gidelim."}))
print(ceviri_zinciri.invoke({"hedef_dil": "Almanca", "metin": "Bugün hava çok güzel, parka gidelim."}))

### Paralel adımlar ve özel fonksiyonlar

- `RunnableParallel`: aynı girdiyle birden fazla zinciri **paralel** çalıştırır, sonuçları sözlükte toplar.
- `RunnableLambda`: herhangi bir Python fonksiyonunu zincirin bir adımına çevirir.
- `RunnablePassthrough`: girdiyi değiştirmeden geçirir.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

ozet_zinciri = ChatPromptTemplate.from_template("Şu metni tek cümleyle özetle:\n{metin}") | llm | StrOutputParser()
anahtar_zinciri = ChatPromptTemplate.from_template("Şu metnin 3 anahtar kelimesini virgülle yaz:\n{metin}") | llm | StrOutputParser()
duygu_zinciri = ChatPromptTemplate.from_template("Şu metnin duygusu nedir? Sadece 'olumlu', 'olumsuz' veya 'nötr' yaz:\n{metin}") | llm | StrOutputParser()

analiz = RunnableParallel(
    ozet=ozet_zinciri,
    anahtar_kelimeler=anahtar_zinciri,
    duygu=duygu_zinciri,
    kelime_sayisi=RunnableLambda(lambda girdi: len(girdi["metin"].split())),
)

yorum = ("Aldığım kablosuz kulaklığın ses kalitesi beklediğimden çok daha iyi çıktı. Pil ömrü de gerçekten "
         "uzun, iki gün boyunca şarj etmeden kullandım. Tek eksisi kutusunun biraz büyük olması.")
for anahtar, deger in analiz.invoke({"metin": yorum}).items():
    print(f"{anahtar:18s}: {deger}")

---
## 5) Yapılandırılmış çıktı (Structured Output)

Uygulamalar genellikle serbest metin değil, **belirli alanları olan veri** ister. `with_structured_output`, bir Pydantic şeması alır ve modelin çıktısını bu şemaya uygun bir Python nesnesine çevirir. Ollama bunu, modelin üretimini JSON şemasına göre **kısıtlayarak** yapar; geçersiz JSON üretilemez.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class UrunYorumu(BaseModel):
    """Bir ürün yorumundan çıkarılan bilgiler."""
    urun: str = Field(description="Yorumlanan ürünün adı veya türü")
    duygu: Literal["olumlu", "olumsuz", "karışık"] = Field(description="Yorumun genel duygusu")
    puan: int = Field(description="Yorumdan tahmin edilen 1-5 arası puan", ge=1, le=5)
    artilar: list[str] = Field(description="Belirtilen olumlu yönler")
    eksiler: list[str] = Field(description="Belirtilen olumsuz yönler")

yapisal_llm = llm.with_structured_output(UrunYorumu)
sonuc = yapisal_llm.invoke(f"Şu yorumu analiz et:\n{yorum}")

print(type(sonuc).__name__)
print(sonuc.model_dump_json(indent=2))

In [ ]:
# Toplu bilgi çıkarma: aynı şemayla birden fazla metin
yorumlar = [
    "Bu blender tam bir hayal kırıklığı, ilk hafta motoru yandı. Müşteri hizmetleri de hiç ilgilenmedi.",
    "Koşu ayakkabısı çok rahat ve hafif, fakat iki ay sonra tabanı aşınmaya başladı.",
    "Kitap muhteşem! Karakterler çok iyi işlenmiş, bir solukta okudum.",
]
import pandas as pd
pd.DataFrame([r.model_dump() for r in yapisal_llm.batch([f"Şu yorumu analiz et:\n{y}" for y in yorumlar])])

---
## 6) Araç (Tool) tanımlama

LLM'ler hesap yapmakta, güncel bilgiye erişmekte veya dış sistemlerde işlem yapmakta zayıftır. **Araçlar**, modelin çağırabileceği Python fonksiyonlarıdır. `@tool` dekoratörü, fonksiyonun **adını**, **docstring**'ini ve **tip ipuçlarını** okuyarak modele gönderilecek bir JSON şeması üretir.

> Docstring ve parametre açıklamaları modelin aracı **ne zaman ve nasıl** kullanacağını anlamasının tek yoludur. İyi yazılmış bir docstring, iyi çalışan bir ajanın yarısıdır.

In [ ]:
import ast
import operator
from datetime import datetime
from langchain_core.tools import tool

_IZINLI_ISLEMLER = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}

def _guvenli_hesapla(dugum):
    if isinstance(dugum, ast.Constant) and isinstance(dugum.value, (int, float)):
        return dugum.value
    if isinstance(dugum, ast.BinOp) and type(dugum.op) in _IZINLI_ISLEMLER:
        return _IZINLI_ISLEMLER[type(dugum.op)](_guvenli_hesapla(dugum.left), _guvenli_hesapla(dugum.right))
    if isinstance(dugum, ast.UnaryOp) and type(dugum.op) in _IZINLI_ISLEMLER:
        return _IZINLI_ISLEMLER[type(dugum.op)](_guvenli_hesapla(dugum.operand))
    raise ValueError("Sadece sayılar ve + - * / ** % işlemleri desteklenir")


@tool
def hesap_makinesi(ifade: str) -> str:
    """Matematiksel bir ifadeyi hesaplar. Toplama, çıkarma, çarpma, bölme, üs (**) ve mod (%) destekler.
    Örnek: '(1250 * 12) * 0.82'"""
    try:
        return str(_guvenli_hesapla(ast.parse(ifade, mode="eval").body))
    except Exception as hata:
        return f"Hata: {hata}"


DOVIZ_KURLARI = {"USD": 41.20, "EUR": 47.85, "GBP": 55.30, "TRY": 1.0}   # örnek (sabit) kurlar

@tool
def doviz_cevir(miktar: float, kaynak: str, hedef: str) -> str:
    """Bir para miktarını bir para biriminden diğerine çevirir. Desteklenen birimler: USD, EUR, GBP, TRY."""
    kaynak, hedef = kaynak.upper(), hedef.upper()
    if kaynak not in DOVIZ_KURLARI or hedef not in DOVIZ_KURLARI:
        return f"Desteklenmeyen para birimi. Desteklenenler: {', '.join(DOVIZ_KURLARI)}"
    sonuc = miktar * DOVIZ_KURLARI[kaynak] / DOVIZ_KURLARI[hedef]
    return f"{miktar:,.2f} {kaynak} = {sonuc:,.2f} {hedef}"


HAVA_DURUMU = {"istanbul": "18°C, parçalı bulutlu", "ankara": "12°C, güneşli", "izmir": "22°C, açık",
               "erzincan": "6°C, karlı", "trabzon": "15°C, yağmurlu"}

@tool
def hava_durumu(sehir: str) -> str:
    """Verilen Türkiye şehri için güncel hava durumunu döndürür."""
    return HAVA_DURUMU.get(sehir.lower().replace("İ", "i"), f"{sehir} için hava durumu verisi bulunamadı.")


@tool
def bugunun_tarihi() -> str:
    """Bugünün tarihini ve haftanın gününü döndürür."""
    gunler = ["Pazartesi", "Salı", "Çarşamba", "Perşembe", "Cuma", "Cumartesi", "Pazar"]
    simdi = datetime.now()
    return f"{simdi:%d.%m.%Y}, {gunler[simdi.weekday()]}"


araclar = [hesap_makinesi, doviz_cevir, hava_durumu, bugunun_tarihi]

In [ ]:
# Modelin göreceği şema
for arac in araclar:
    print(f"🔧 {arac.name}: {arac.description.splitlines()[0]}")
    print(f"   parametreler: {arac.args}\n")

# Araçlar normal fonksiyon gibi de çağrılabilir
print(hesap_makinesi.invoke({"ifade": "(1250 * 12) * 0.82"}))
print(doviz_cevir.invoke({"miktar": 100, "kaynak": "eur", "hedef": "try"}))

---
## 7) Tool calling: döngüyü elle yazmak

**Önemli:** Model aracı **kendisi çalıştırmaz**. Sadece "şu aracı şu argümanlarla çağır" diye bir **istek** üretir. Aracı çalıştırıp sonucu modele geri vermek bizim (veya ajan çatısının) işidir:

```
1. Kullanıcı mesajı + araç şemaları ──► LLM
2. LLM ──► AIMessage(tool_calls=[{name: "doviz_cevir", args: {...}, id: "abc"}])
3. Biz aracı çalıştırırız ──► ToolMessage(content="100 EUR = 4785 TRY", tool_call_id="abc")
4. Tüm geçmiş ──► LLM ──► (başka araç çağrısı VEYA nihai cevap)
5. Araç çağrısı kalmayana kadar 2–4 tekrarlanır
```

In [ ]:
llm_araclarla = llm.bind_tools(araclar)

ai_mesaji = llm_araclarla.invoke("250 dolar kaç TL eder?")
print("İçerik    :", repr(ai_mesaji.content))
print("Araç çağrısı:", ai_mesaji.tool_calls)

In [ ]:
arac_haritasi = {a.name: a for a in araclar}

def arac_dongusu(soru, maks_adim=6, ayrintili=True):
    mesajlar = [SystemMessage("Sen yardımcı bir asistansın. Gerektiğinde araçları kullan. Türkçe cevap ver."),
                HumanMessage(soru)]
    for adim in range(maks_adim):
        ai = llm_araclarla.invoke(mesajlar)
        mesajlar.append(ai)
        if not ai.tool_calls:                                  # araç çağrısı yok → nihai cevap
            return ai.content
        for cagri in ai.tool_calls:                            # model birden fazla aracı paralel isteyebilir
            sonuc = arac_haritasi[cagri["name"]].invoke(cagri["args"])
            if ayrintili:
                print(f"  🔧 {cagri['name']}({cagri['args']}) → {sonuc}")
            mesajlar.append(ToolMessage(content=str(sonuc), tool_call_id=cagri["id"]))
    return "Maksimum adım sayısına ulaşıldı."

for soru in ["İstanbul ve Erzincan'da hava nasıl?",
             "Aylık maaşım 1850 euro. Yıllık kaç TL eder? %18 vergi düşünce ne kalır?",
             "Bugün günlerden ne?"]:
    print(f"\n❓ {soru}")
    print(f"💬 {arac_dongusu(soru)}")

---
## 8) Ajan: `create_agent`

Yukarıdaki döngü her ajanın kalbidir. LangChain bunu hazır olarak sunar: `create_agent`, arka planda bir **LangGraph** grafı kurar (model düğümü ⇄ araç düğümü) ve döngüyü bizim yerimize yönetir. Ayrıca hafıza, ara katman (middleware), yapılandırılmış nihai cevap gibi özellikler ekler.

In [ ]:
from langchain.agents import create_agent

ajan = create_agent(
    model=llm,
    tools=araclar,
    system_prompt="Sen bir seyahat ve finans asistanısın. Hesaplamalar için mutlaka hesap_makinesi aracını kullan. Türkçe cevap ver.",
)

def ajana_sor(ajan, soru, config=None):
    sonuc = ajan.invoke({"messages": [{"role": "user", "content": soru}]}, config=config)
    for m in sonuc["messages"]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for c in m.tool_calls:
                print(f"  🔧 {c['name']}({c['args']})")
        elif isinstance(m, ToolMessage):
            print(f"  ↳ {m.content}")
    print(f"💬 {sonuc['messages'][-1].content}")

ajana_sor(ajan, "Hafta sonu İzmir'e gideceğim. Hava nasıl olacak? Ayrıca 3 gece için gecelik 85 euroluk otelin toplam TL tutarı ne?")

---
## 9) Hafıza: çok turlu sohbet

Ajan varsayılan olarak **durumsuzdur**; her `invoke` sıfırdan başlar. Bir **checkpointer** eklersek her konuşmanın (thread) durumu kaydedilir ve aynı `thread_id` ile devam edilebilir.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

hafizali_ajan = create_agent(model=llm, tools=araclar, checkpointer=InMemorySaver(),
                             system_prompt="Sen yardımcı bir asistansın. Türkçe cevap ver.")

konusma = {"configurable": {"thread_id": "berk-1"}}
for mesaj in ["Merhaba, benim adım Berk ve Ankara'da yaşıyorum.",
              "Yaşadığım şehirde hava nasıl?",
              "Adımı hatırlıyor musun?"]:
    print(f"\n👤 {mesaj}")
    ajana_sor(hafizali_ajan, mesaj, config=konusma)

print("\n--- Farklı bir thread_id = sıfırdan konuşma ---")
ajana_sor(hafizali_ajan, "Adımı hatırlıyor musun?", config={"configurable": {"thread_id": "yeni-kullanici"}})

---
## 10) RAG zinciri ve retriever aracı

LangChain'in RAG için hazır bileşenleri:

| Bileşen | Görevi | Burada kullandığımız |
|---|---|---|
| Text splitter | Dokümanı parçalara böler | `RecursiveCharacterTextSplitter` |
| Embeddings | Metni vektöre çevirir | `HuggingFaceEmbeddings` (çok dilli, ücretsiz) |
| Vector store | Vektörleri saklar, benzerlik araması yapar | `InMemoryVectorStore` (üretimde: Chroma, FAISS, Qdrant, pgvector) |
| Retriever | Soru → ilgili parçalar | `vector_store.as_retriever()` |

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

el_kitabi = """
Kuzey Yıldızı Yazılım'da esnek mesai uygulanır. Çekirdek çalışma saatleri 10:00 ile 16:00 arasıdır.
Çalışanlar haftada en fazla 3 gün uzaktan çalışabilir. Salı günleri tüm ekiplerin ofiste bulunması zorunludur.

Tam zamanlı çalışanların yıllık izin hakkı ilk üç yıl için 18 iş günüdür. Üç yılı dolduranların izni 24 iş gününe çıkar.
Kullanılmayan izinin en fazla 5 günü bir sonraki yıla devredilebilir.

Her çalışanın yıllık kişisel gelişim bütçesi 15.000 TL'dir. Bu bütçe kurs, sertifika ve konferans için kullanılabilir.
Ofiste çalışılan her gün için yemek kartına günlük 450 TL yüklenir. Uzaktan çalışılan günlerde yükleme yapılmaz.

Şirket ağına dışarıdan bağlanmak için KYVPN uygulaması kullanılır. VPN şifresi 90 günde bir değiştirilmelidir.
Kayıp veya çalıntı cihazlar en geç 1 saat içinde güvenlik ekibine bildirilmelidir.
"""

bolucu = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
parcalar = bolucu.split_documents([Document(page_content=el_kitabi, metadata={"kaynak": "el_kitabi"})])
print(f"{len(parcalar)} parça oluştu. Örnek: {parcalar[0].page_content!r}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vektor_deposu = InMemoryVectorStore.from_documents(parcalar, embeddings)
retriever = vektor_deposu.as_retriever(search_kwargs={"k": 2})

for d in retriever.invoke("Evden kaç gün çalışabilirim?"):
    print("  📄", d.page_content.replace("\n", " "))

### LCEL ile RAG zinciri

In [ ]:
rag_istemi = ChatPromptTemplate.from_messages([
    ("system", "Soruyu YALNIZCA aşağıdaki bağlama dayanarak Türkçe ve kısa cevapla. "
               "Bağlamda cevap yoksa 'Bu bilgi el kitabında yok.' de.\n\nBağlam:\n{baglam}"),
    ("human", "{soru}"),
])

def parcalari_birlestir(dokumanlar):
    return "\n\n".join(d.page_content for d in dokumanlar)

rag_zinciri = (
    {"baglam": retriever | parcalari_birlestir, "soru": RunnablePassthrough()}
    | rag_istemi
    | llm
    | StrOutputParser()
)

for soru in ["Yıllık izin hakkım kaç gün?", "Evden çalıştığım günlerde yemek kartına para yüklenir mi?",
             "Şirket servisi var mı?"]:
    print(f"❓ {soru}\n💬 {rag_zinciri.invoke(soru)}\n")

### Retriever'ı ajanın aracı yapmak

Zincir her soruda arama yapar. Ajan ise **ne zaman** arama yapacağına kendisi karar verir ve aramayı diğer araçlarla birleştirebilir: önce el kitabından bilgiyi bul, sonra hesap makinesiyle hesapla.

In [ ]:
@tool
def el_kitabinda_ara(sorgu: str) -> str:
    """Kuzey Yıldızı Yazılım çalışan el kitabında arama yapar. Mesai, uzaktan çalışma, izin, eğitim bütçesi,
    yemek kartı, VPN ve cihaz güvenliği gibi şirket kuralları hakkındaki sorular için kullan."""
    return parcalari_birlestir(retriever.invoke(sorgu))

ik_ajani = create_agent(
    model=llm,
    tools=[el_kitabinda_ara, hesap_makinesi],
    system_prompt=("Sen Kuzey Yıldızı Yazılım'ın İK asistanısın. Şirket kurallarını ASLA tahmin etme, "
                   "her zaman el_kitabinda_ara aracıyla kontrol et. Hesaplamalar için hesap_makinesi kullan. Türkçe cevap ver."),
)
ajana_sor(ik_ajani, "Bu ay 22 iş günü var ve haftada 3 gün evden çalışacağım. Bu ay yemek kartıma yaklaşık ne kadar yüklenir?")

---
## Özet

| Kavram | LangChain karşılığı |
|---|---|
| Model çağrısı | `ChatOllama(...).invoke / stream / batch` |
| İstem şablonu | `ChatPromptTemplate.from_messages([...])` |
| Zincir | LCEL: `prompt \| llm \| StrOutputParser()` |
| Paralel adımlar | `RunnableParallel`, `RunnableLambda`, `RunnablePassthrough` |
| Yapılandırılmış çıktı | `llm.with_structured_output(PydanticModeli)` |
| Araç | `@tool` + docstring + tip ipuçları |
| Tool calling | `llm.bind_tools(araclar)` → `AIMessage.tool_calls` → `ToolMessage` |
| Ajan | `create_agent(model, tools, system_prompt)` |
| Hafıza | `checkpointer=InMemorySaver()` + `thread_id` |
| RAG | splitter → embeddings → vector store → retriever → zincir / araç |

**İyi uygulamalar:**
- Araç docstring'lerini modelin okuyacağını unutmadan yaz: ne yapar, ne zaman kullanılır, örnek girdi.
- Araçlar hata fırlatmak yerine **açıklayıcı hata mesajı döndürsün**; model bunu okuyup düzeltebilir.
- Geri alınamaz işlemler yapan araçlarda (e-posta, ödeme, silme) **insan onayı** iste (→ `03_langgraph_coklu_ajan.ipynb`).
- Ajan döngüsüne **maksimum adım** sınırı koy.
- Araç sonuçları ve getirilen dokümanlar **güvenilmeyen veridir** (→ `transformers-llm/09_llm_guvenligi.md`).

Bir sonraki defter: **LangGraph** ile kendi ajan grafiklerimizi, insan onayını ve çoklu ajan sistemlerini kuruyoruz.